In [1]:
from __future__ import annotations

import gc
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


@dataclass
class CFG:
    seed: int = 42
    input_dir: str = "/kaggle/input/competitions/nlp-getting-started"
    output_dir: str = "/kaggle/working"
    model_name: str = "roberta-base"
    max_length: int = 128
    valid_size: float = 0.2
    epochs: int = 4
    train_batch_size: int = 16
    valid_batch_size: int = 32
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    use_keyword: bool = True
    use_location: bool = False
    ensemble_alpha_transformer: float = 0.8




def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def is_missing(value) -> bool:
    return value is None or value != value or str(value).strip() == ""


def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)
    text = HTML_RE.sub(" ", text)
    text = text.replace("#", " ")
    text = SPACE_RE.sub(" ", text)
    return text.strip()


def build_text(row: dict) -> str:
    pieces = []
    if cfg.use_keyword and not is_missing(row.get("keyword")):
        pieces.append(f"keyword: {row['keyword']}")
    if cfg.use_location and not is_missing(row.get("location")):
        pieces.append(f"location: {row['location']}")
    pieces.append(str(row.get("text", "")))
    return clean_text(" ".join(pieces))





In [2]:
cfg = CFG()



URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"&[a-z]+;")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


In [3]:
def metrics(y_true, y_pred, y_score=None) -> dict[str, float]:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        y_score = np.nan_to_num(y_score, nan=0.5, posinf=1.0, neginf=0.0)
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    return {k: float(v) for k, v in out.items()}


def find_best_threshold(y_true, scores, start=0.0, stop=1.0, steps=401) -> tuple[float, float]:
    scores = np.nan_to_num(scores, nan=0.5, posinf=1.0, neginf=0.0)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in np.linspace(start, stop, steps):
        preds = (scores >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_threshold = float(threshold)
            best_f1 = float(score)
    return best_threshold, best_f1


class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def collate_fn(batch, tokenizer, max_length):
    texts = [item["text"] for item in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    if "label" in batch[0]:
        enc["labels"] = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return enc


def train_tfidf(train_df, valid_df):
    model = Pipeline(
        [
            (
                "features",
                FeatureUnion(
                    [
                        (
                            "word",
                            TfidfVectorizer(
                                analyzer="word",
                                max_features=50000,
                                ngram_range=(1, 2),
                                min_df=2,
                                sublinear_tf=True,
                            ),
                        ),
                        (
                            "char",
                            TfidfVectorizer(
                                analyzer="char_wb",
                                ngram_range=(3, 5),
                                min_df=2,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            ("clf", LogisticRegression(C=2.0, max_iter=2000, solver="liblinear", random_state=cfg.seed)),
        ]
    )
    model.fit(train_df["clean_text"], train_df["target"])
    valid_scores = model.predict_proba(valid_df["clean_text"])[:, 1]
    threshold, _ = find_best_threshold(valid_df["target"].values, valid_scores)
    valid_preds = (valid_scores >= threshold).astype(int)
    print("TF-IDF metrics:", json.dumps(metrics(valid_df["target"], valid_preds, valid_scores), indent=2))
    print("TF-IDF threshold:", threshold)
    return model, valid_scores, threshold

In [4]:
def train_transformer(train_df, valid_df):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2).to(device)

    train_loader = DataLoader(
        TweetDataset(train_df["clean_text"], train_df["target"]),
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_fn(batch, tokenizer, cfg.max_length),
    )
    valid_loader = DataLoader(
        TweetDataset(valid_df["clean_text"], valid_df["target"]),
        batch_size=cfg.valid_batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, tokenizer, cfg.max_length),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best = {"f1": -1.0, "state": None, "threshold": 0.5, "scores": None}
    for epoch in range(cfg.epochs):
        model.train()
        losses = []
        nan_batches = 0
        progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{cfg.epochs}")
        for batch in progress:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            out = model(**batch)
            if not torch.isfinite(out.loss):
                nan_batches += 1
                if nan_batches >= 3:
                    raise RuntimeError(
                        "Training became numerically unstable: loss is NaN. "
                        "Restart the session and use roberta-base/distilbert-base-uncased "
                        "or lower learning_rate."
                    )
                continue
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(out.loss.item())
            progress.set_postfix(loss=np.mean(losses[-50:]))

        valid_scores = predict_transformer_scores(model, tokenizer, valid_df["clean_text"].tolist(), device)
        valid_scores = np.nan_to_num(valid_scores, nan=0.5, posinf=1.0, neginf=0.0)
        threshold, valid_f1 = find_best_threshold(valid_df["target"].values, valid_scores)
        valid_preds = (valid_scores >= threshold).astype(int)
        valid_metrics = metrics(valid_df["target"], valid_preds, valid_scores)
        print(f"epoch {epoch + 1} metrics:", json.dumps(valid_metrics, indent=2), "threshold:", threshold)

        if valid_f1 > best["f1"]:
            best = {
                "f1": valid_f1,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "threshold": threshold,
                "scores": valid_scores,
            }

    model.load_state_dict(best["state"])
    return model, tokenizer, best


def predict_transformer_scores(model, tokenizer, texts, device):
    loader = DataLoader(
        TweetDataset(texts),
        batch_size=cfg.valid_batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, tokenizer, cfg.max_length),
    )
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="predict"):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            scores.extend(probs.detach().cpu().numpy().tolist())
    return np.array(scores)


In [5]:
seed_everything(cfg.seed)
input_dir = Path(cfg.input_dir)
output_dir = Path(cfg.output_dir)
train = pd.read_csv(input_dir / "train.csv")
test = pd.read_csv(input_dir / "test.csv")
train["clean_text"] = [build_text(row) for row in train.to_dict(orient="records")]
test["clean_text"] = [build_text(row) for row in test.to_dict(orient="records")]

train_df, valid_df = train_test_split(
    train,
    test_size=cfg.valid_size,
    stratify=train["target"],
    random_state=cfg.seed,
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

tfidf_model, tfidf_valid_scores, tfidf_threshold = train_tfidf(train_df, valid_df)
transformer_model, tokenizer, best = train_transformer(train_df, valid_df)

y_valid = valid_df["target"].values
best_ensemble = None
for alpha in np.linspace(0, 1, 21):
    scores = alpha * best["scores"] + (1 - alpha) * tfidf_valid_scores
    threshold, _ = find_best_threshold(y_valid, scores)
    preds = (scores >= threshold).astype(int)
    result = metrics(y_valid, preds, scores)
    result["alpha_transformer"] = float(alpha)
    result["threshold"] = threshold
    if best_ensemble is None or result["f1"] > best_ensemble["f1"]:
        best_ensemble = result
print("Best ensemble:", json.dumps(best_ensemble, indent=2))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_transformer_scores = predict_transformer_scores(transformer_model, tokenizer, test["clean_text"].tolist(), device)
test_tfidf_scores = tfidf_model.predict_proba(test["clean_text"])[:, 1]
test_scores = (
    best_ensemble["alpha_transformer"] * test_transformer_scores
    + (1 - best_ensemble["alpha_transformer"]) * test_tfidf_scores
)
test_preds = (test_scores >= best_ensemble["threshold"]).astype(int)

submission = pd.DataFrame({"id": test["id"], "target": test_preds})
submission_path = output_dir / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
print(submission.head())

with (output_dir / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "model_name": cfg.model_name,
            "transformer_best_f1": best["f1"],
            "transformer_threshold": best["threshold"],
            "ensemble": best_ensemble,
        },
        file,
        indent=2,
    )
gc.collect()
torch.cuda.empty_cache()


TF-IDF metrics: {
  "accuracy": 0.8227183191070256,
  "precision": 0.8127035830618893,
  "recall": 0.7629969418960245,
  "f1": 0.7870662460567823,
  "roc_auc": 0.8768136597656979
}
TF-IDF threshold: 0.48
Device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

epoch 4/4:   0%|          | 0/381 [00:00<?, ?it/s]

predict:   0%|          | 0/48 [00:00<?, ?it/s]

epoch 4 metrics: {
  "accuracy": 0.8562048588312541,
  "precision": 0.8361669242658424,
  "recall": 0.827217125382263,
  "f1": 0.8316679477325134,
  "roc_auc": 0.9064304642054033
} threshold: 0.3925
Best ensemble: {
  "accuracy": 0.8562048588312541,
  "precision": 0.8361669242658424,
  "recall": 0.827217125382263,
  "f1": 0.8316679477325134,
  "roc_auc": 0.9080079039143027,
  "alpha_transformer": 0.9,
  "threshold": 0.3875
}


predict:   0%|          | 0/102 [00:00<?, ?it/s]

Saved: /kaggle/working/submission.csv
   id  target
0   0       1
1   2       1
2   3       1
3   9       1
4  11       1
